<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Chapter 17 · Asset Management Foundations
&copy; Dr. Yves J. Hilpisch<br>
AI-supported by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook collects executable versions of the core Chapter 17 examples:
- tracking error for a simple equity–bond portfolio vs `SPY`,
- a cross-sectional holdings snapshot and sector weights, and
- basic panel-style price data structures and a small universe table.


### Imports
We start with the standard numerical and data-analysis libraries used
throughout the book.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
# Configure Matplotlib defaults for the book
mpl.style.use("seaborn-v0_8")  # baseline plotting style
mpl.rcParams.update({"font.family": "serif"})
mpl.rcParams.update({"figure.dpi": 300})


### Tracking Error Example
Compute annualised tracking error for an equally weighted portfolio of `AAPL`,
`JPM`, and `TLT` versus `SPY` over roughly two years of daily data.


In [ ]:
LOCAL_EOD = Path("..") / "data" / "eod_data.csv"
REMOTE_EOD = "https://hilpisch.com/eod_data.csv"
source = LOCAL_EOD if LOCAL_EOD.exists() else REMOTE_EOD

prices = pd.read_csv(
    source,
    parse_dates=["Date"],
    index_col="Date",
)

symbols = ["AAPL", "JPM", "TLT"]
sub = prices[symbols + ["SPY"]]

rets_all = sub.pct_change().dropna()
rets = rets_all.iloc[-2 * 252 :]

w_eq = np.repeat(1.0 / len(symbols), len(symbols))
r_port = (rets[symbols] * w_eq).sum(axis=1)
r_bench = rets["SPY"]
active = r_port - r_bench

te_annual = active.std(ddof=1) * np.sqrt(252.0)
te_annual

### Cross-Sectional Holdings Snapshot
We next build a small holdings table with quantities, prices, sectors, and
regions, and compute market values and portfolio weights that will feed into
simple exposure and visualisation examples.


In [ ]:
holdings = pd.DataFrame(
    {
        "symbol": ["AAPL", "NVDA", "JPM", "SPY"],
        "quantity": [120, 80, 150, 200],
        "price": [180.25, 820.10, 145.30, 520.10],
        "sector": ["Technology", "Technology", "Financials", "Equity Index"],
        "region": ["US", "US", "US", "Global"],
        "currency": ["USD", "USD", "USD", "USD"],
    }
)
holdings["market_value"] = holdings["quantity"] * holdings["price"]
holdings["weight"] = holdings["market_value"] / holdings["market_value"].sum()
holdings

In [ ]:
sector_weights = holdings.groupby("sector")["weight"].sum()
sector_weights

### Wide and Long Price Tables
These cells show how to move from wide price tables to a long `MultiIndex`
representation using `.stack()`, mirroring the panel-style structures used
later for returns and holdings histories.


In [ ]:
prices_small = prices[["AAPL", "NVDA", "JPM", "SPY"]]
stacked = prices_small.stack().to_frame("price")
stacked.index.names = ["Date", "symbol"]
stacked.head(8)

### Universe Metadata Table
Create a small universe table keyed by `instrument_id` with basic attributes.


In [ ]:
universe = pd.DataFrame(
    {
        "instrument_id": [1, 2, 3, 4],
        "symbol": ["AAPL", "NVDA", "JPM", "SPY"],
        "asset_class": ["Equity", "Equity", "Equity", "Equity ETF"],
        "region": ["US", "US", "US", "Global"],
        "esg_flag": [False, False, False, False],
    }
).set_index("instrument_id")
universe

The next cells reproduce the chapter figures for the holdings snapshot
and tracking-error illustration.


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.0))
ax.bar(holdings["symbol"], holdings["weight"])
ax.set_ylabel("Portfolio weight")
ax.set_title("Illustrative Cross-Sectional Portfolio Weights")
ax.grid(True, axis="y", linestyle="--", alpha=0.3)
for x, w in zip(holdings["symbol"], holdings["weight"]):
    ax.text(x, w, f"{w:.1%}", ha="center", va="bottom", fontsize=8)
fig.tight_layout()

In [ ]:
equity = (1 + r_port).cumprod()
bench = (1 + r_bench).cumprod()
active = equity - bench

fig, (ax_top, ax_bottom) = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
ax_top.plot(equity.index, equity, label="Portfolio")
ax_top.plot(bench.index, bench, label="SPY")
ax_top.set_ylabel("Cumulative value (normalised)")
ax_top.grid(True, linestyle="--", alpha=0.3)
ax_top.legend(loc="upper left")

ax_bottom.plot(active.index, active, label="Active return (portfolio − SPY)")
ax_bottom.axhline(0.0, color="black", linewidth=0.8, alpha=0.8)
ax_bottom.set_ylabel("Active return (daily)")
ax_bottom.set_xlabel("Date")
ax_bottom.grid(True, linestyle="--", alpha=0.3)
ax_bottom.legend(loc="upper left")
fig.tight_layout()

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
